In [ ]:
validate_dataframe(customers, "Customers")
validate_dataframe(orders, "Orders")
validate_dataframe(sessions, "Sessions")

In [ ]:
customers = pd.read_csv("../data/processed/customers.csv")
orders = pd.read_csv("../data/processed/orders.csv")
sessions = pd.read_csv("../data/processed/sessions.csv")

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../"))

from src.validation_engine import validate_dataframe
import pandas as pd

# A/B Experimentation Analysis

This notebook evaluates the impact of a pricing experiment conducted on the simulated e-commerce platform.

Customers were randomly assigned into two groups:

Control Group → Low discount range  
Treatment Group → Higher discount range

The objective is to understand whether larger discounts improve conversion while maintaining profitability.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Load experiment data
customers = pd.read_csv('../data/processed/customer_latent_traits.csv')
orders = pd.read_csv('../data/raw/orders.csv')

# Create experiment groups (50/50 split based on customer_id)
customers['experiment_group'] = np.where(customers['customer_id'] % 2 == 0, 'control', 'treatment')

# Merge data for analysis
experiment_data = orders.merge(customers[['customer_id', 'experiment_group']], on='customer_id', how='left')

# Use existing calculated fields from orders data
experiment_data['revenue'] = experiment_data['final_price']
experiment_data['cost'] = experiment_data['cost']  # cost from orders
experiment_data['margin'] = experiment_data['margin']  # margin already calculated

print("Experiment data shape:", experiment_data.shape)
print("\nExperiment groups distribution:")
print(experiment_data['experiment_group'].value_counts())

KeyError: 'cost_x'

In [ ]:
# Conversion Analysis
conversion_by_group = experiment_data.groupby('experiment_group').agg({
    'customer_id': 'nunique',
    'order_id': 'count'
}).reset_index()

conversion_by_group.columns = ['experiment_group', 'unique_customers', 'total_orders']
conversion_by_group['conversion_rate'] = conversion_by_group['total_orders'] / conversion_by_group['unique_customers']

print("Conversion Analysis:")
print(conversion_by_group)

# Statistical test for conversion
control_conv = conversion_by_group[conversion_by_group['experiment_group'] == 'control']['conversion_rate'].iloc[0]
treatment_conv = conversion_by_group[conversion_by_group['experiment_group'] == 'treatment']['conversion_rate'].iloc[0]

# Chi-square test for proportions
from statsmodels.stats.proportion import proportions_ztest

control_orders = conversion_by_group[conversion_by_group['experiment_group'] == 'control']['total_orders'].iloc[0]
treatment_orders = conversion_by_group[conversion_by_group['experiment_group'] == 'treatment']['total_orders'].iloc[0]
control_cust = conversion_by_group[conversion_by_group['experiment_group'] == 'control']['unique_customers'].iloc[0]
treatment_cust = conversion_by_group[conversion_by_group['experiment_group'] == 'treatment']['unique_customers'].iloc[0]

z_stat, p_value = proportions_ztest([control_orders, treatment_orders], [control_cust, treatment_cust])

print(f"\nConversion Rate Test:")
print(f"Control: {control_conv:.3f}")
print(f"Treatment: {treatment_conv:.3f}")
print(f"Relative Increase: {((treatment_conv - control_conv) / control_conv * 100):.1f}%")
print(f"Z-statistic: {z_stat:.3f}, P-value: {p_value:.3f}")

In [ ]:
# Revenue and Margin Analysis
revenue_margin_by_group = experiment_data.groupby('experiment_group').agg({
    'revenue': 'sum',
    'margin': 'sum',
    'customer_id': 'nunique'
}).reset_index()

revenue_margin_by_group['avg_revenue_per_customer'] = revenue_margin_by_group['revenue'] / revenue_margin_by_group['customer_id']
revenue_margin_by_group['avg_margin_per_customer'] = revenue_margin_by_group['margin'] / revenue_margin_by_group['customer_id']
revenue_margin_by_group['margin_percentage'] = (revenue_margin_by_group['margin'] / revenue_margin_by_group['revenue']) * 100

print("Revenue and Margin Analysis:")
print(revenue_margin_by_group.round(2))

# Statistical tests for revenue and margin
control_revenue = experiment_data[experiment_data['experiment_group'] == 'control']['revenue']
treatment_revenue = experiment_data[experiment_data['experiment_group'] == 'treatment']['revenue']

control_margin = experiment_data[experiment_data['experiment_group'] == 'control']['margin']
treatment_margin = experiment_data[experiment_data['experiment_group'] == 'treatment']['margin']

# T-tests
revenue_t_stat, revenue_p_value = stats.ttest_ind(control_revenue, treatment_revenue)
margin_t_stat, margin_p_value = stats.ttest_ind(control_margin, treatment_margin)

print(f"\nRevenue per Customer Test:")
print(f"Control: ${revenue_margin_by_group[revenue_margin_by_group['experiment_group'] == 'control']['avg_revenue_per_customer'].iloc[0]:.2f}")
print(f"Treatment: ${revenue_margin_by_group[revenue_margin_by_group['experiment_group'] == 'treatment']['avg_revenue_per_customer'].iloc[0]:.2f}")
print(f"T-statistic: {revenue_t_stat:.3f}, P-value: {revenue_p_value:.3f}")

print(f"\nMargin per Customer Test:")
print(f"Control: ${revenue_margin_by_group[revenue_margin_by_group['experiment_group'] == 'control']['avg_margin_per_customer'].iloc[0]:.2f}")
print(f"Treatment: ${revenue_margin_by_group[revenue_margin_by_group['experiment_group'] == 'treatment']['avg_margin_per_customer'].iloc[0]:.2f}")
print(f"T-statistic: {margin_t_stat:.3f}, P-value: {margin_p_value:.3f}")

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Conversion Rate Comparison
conversion_by_group.plot(kind='bar', x='experiment_group', y='conversion_rate', ax=axes[0,0], color=['skyblue', 'lightcoral'])
axes[0,0].set_title('Conversion Rate by Experiment Group')
axes[0,0].set_ylabel('Conversion Rate')
axes[0,0].tick_params(axis='x', rotation=0)

# Average Revenue per Customer
revenue_margin_by_group.plot(kind='bar', x='experiment_group', y='avg_revenue_per_customer', ax=axes[0,1], color=['skyblue', 'lightcoral'])
axes[0,1].set_title('Average Revenue per Customer')
axes[0,1].set_ylabel('Revenue ($)')
axes[0,1].tick_params(axis='x', rotation=0)

# Average Margin per Customer
revenue_margin_by_group.plot(kind='bar', x='experiment_group', y='avg_margin_per_customer', ax=axes[1,0], color=['skyblue', 'lightcoral'])
axes[1,0].set_title('Average Margin per Customer')
axes[1,0].set_ylabel('Margin ($)')
axes[1,0].tick_params(axis='x', rotation=0)

# Margin Percentage
revenue_margin_by_group.plot(kind='bar', x='experiment_group', y='margin_percentage', ax=axes[1,1], color=['skyblue', 'lightcoral'])
axes[1,1].set_title('Margin Percentage by Group')
axes[1,1].set_ylabel('Margin %')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# Conclusions

## Key Findings:

1. **Conversion Impact**: The treatment group (higher discounts) showed a [X]% increase in conversion rate compared to control (p-value: [Y]).

2. **Revenue Impact**: Average revenue per customer [increased/decreased] by [Z]% in the treatment group.

3. **Margin Impact**: Despite higher discounts, margins per customer [increased/decreased] by [W]%, suggesting the volume increase offset the discount impact.

## Business Recommendations:

- **If conversion increase is significant and margins are maintained**: Scale the higher discount strategy
- **If margins decline significantly**: Consider targeted discounting or alternative growth strategies
- **Further analysis needed**: Customer segmentation analysis to identify which customer types respond best to discounts

## Statistical Significance:
- Conversion rate difference: [Significant/Not significant]
- Revenue difference: [Significant/Not significant]
- Margin difference: [Significant/Not significant]

This analysis demonstrates the importance of testing pricing strategies while monitoring both customer acquisition and profitability metrics.